In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import re

from sklearn.pipeline import Pipeline
from sklearn.cluster import DBSCAN
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier

In [2]:
log_files = glob.glob('/tmp/*.log')

In [20]:
    
common_messages = [
    r'Accepted',
    r'pam_unix\([a-zA-Z!@#$%^&*()+\-={}[\]|\\:;"<>,.?/~`]+\:\w+\)',
    r'Failed',
    r'Received',
    r'Invalid user',
    r'Authentication failure',
    r'refused connect from',
    r'error: maximum authentication attempts exceeded',
    r'Connection closed by',
    r'New session',
    r'Disconnected',
    r'PAM 2 more'
]


cm_pattern = r'\s({})'.format('|'.join(common_messages))

pattern_step1 = r'(\w+\s\d+\s\d{2}:\d{2}:\d{2})\s(\w+)\s(\w+)(\[(\d+)\]|):' + cm_pattern

messages = [
    r'(pam_unix\(sshd:session\): session\s(\w+)\s\w+\s\w+\s(\w+)(\((\w+=\d+)\)\s\w+\s\((\w+=\d+)\)|))',
    r'((Invalid\suser|Disconnected\sfrom\sinvalid\suser|Disconnected\sfrom\sauthenticating\suser)\s(\w+[a-zA-Z!@#$%^&*()+\-={}[\]|\\:;"<>,.?/~`]+|None)(\s\w+|)\s(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s\w+\s(\d+))',
    r'((pam_unix\(sshd:auth\):|pam_unix\(proxmox-ve-auth:auth\):|PAM 2 more)\s\w+\s(\w+);\s\w+=(\w+|)\s\w+=(\d+|)\s\w+=(\d+|)\s\w+=(\w+|)\s\w+=(\w+|)\s\w+=(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s+(\w+=(\w+|)|))',
    r'(Received\s(\w+)\s\w+\s(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s\w+\s(\d+))'
    r'(Accepted\s(\w+)\s\w+\s(\w+)\s\w+\s(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s\w+\s(\d+)\s(\w+)(:\s(\w+\s\w+):([a-zA-Z0-9!@#$%^&*()+\-={}[\]|\\:;"<>,.?/~`]+)|))'
]


pattern_step2 = r'{}'.format('|'.join(messages))
log_dict = {}
log_list = []

for file_path in log_files:
    with open(file_path, 'r') as file:    
    # Lire chaque ligne du fichier auth.log
        for line in file.readlines():
            print(line)           
            # Rechercher les correspondances dans chaque ligne avec le modèle d'expression régulière
            match_step1 = re.search(pattern_step1, line)
            if match_step1:
                # Extraire les informations spécifiques à l'aide des groupes capturés
                log_dict = {
                    'Timestamp': match_step1.group(1),
                    'Node' : match_step1.group(2),
                    'Event' : match_step1.group(3),
                    'sshd_id' : match_step1.group(5),
                    'common_messages' : match_step1.group(6),
                    'type_session': None,
                    'type_accepted': None,
                    'type_received': None,
                    'type_connection': None,
                    'type_crypto' : None,
                    'public_key' : None,
                    'type_fail': None,
                    'type_auth': None,
                    'user': None,
                    'IP_user' : None,
                    'port_user': None,
                    'logname': None,
                    'uid_target': None,
                    'uid_init' : None,
                    'euid' : None,
                    'tty' : None,
                    'ruser' : None,
                    'rhost': None,
                    'others': None
                }
            match_step2 = re.search(pattern_step2, line)
            if match_step2:
                #print("common_messages:", log_dict['common_messages'])
                if log_dict.get('common_messages') == 'pam_unix(sshd:session)' :
                    log_dict['type_session'] = match_step2.group(2)
                    log_dict['user'] = match_step2.group(3)
                    log_dict['uid_target'] = match_step2.group(5)
                    log_dict['uid_init'] = match_step2.group(6)
                elif log_dict.get('common_messages') == 'Invalid user' or log_dict.get('common_messages') == 'Disconnected':
                    print('here invalid| disconnected')
                    log_dict['user'] = match_step2.group(9)
                    log_dict['IP_user'] = match_step2.group(11)
                    log_dict['port_user'] = match_step2.group(12)
                    """print("Username:", log_dict['user'])
                    print("IP Address:", log_dict['IP_user'])
                    print("Port:", log_dict['port_user'])"""
                elif log_dict.get('common_messages') == 'pam_unix(sshd:auth)' or log_dict.get('common_messages') == 'PAM 2 more' or log_dict.get('common_messages') == 'pam_unix(proxmox-ve-auth:auth)':
                    log_dict['type_auth'] = match_step2.group(15)
                    log_dict['logname'] = match_step2.group(16)
                    log_dict['uid_init'] = match_step2.group(17)
                    log_dict['euid'] = match_step2.group(18)
                    log_dict['tty'] = match_step2.group(19)
                    log_dict['ruser'] = match_step2.group(20)
                    log_dict['rhost'] = match_step2.group(21)
                    log_dict['user'] = match_step2.group(23)
                elif log_dict.get('common_messages') == 'Received' :
                    log_dict['type_received'] = match_step2.group(25)
                    log_dict['IP_user'] = match_step2.group(26)
                    log_dict['port_user'] = match_step2.group(27)
                elif log_dict.get('common_messages') == 'Accepted' :
                    log_dict['type_accepted'] = match_step2.group(29)
                    log_dict['user'] = match_step2.group(30)
                    log_dict['IP_user'] = match_step2.group(31)
                    log_dict['port_user'] = match_step2.group(32)
                    log_dict['type_connection'] = match_step2.group(33)
                    log_dict['type_crypto'] = match_step2.group(35)
                    log_dict['public_key'] = match_step2.group(36)
                else :
                    log_dict['others'] = match_step2.group(0)
            print(log_dict)        
        # Entraînez le modèle
        #pipeline_model.fit(X, y)
        
        """#logs = file.read()
        # Traitez les logs selon vos besoins
        # ...
        matches = []
        # Lire chaque ligne du fichier journal
        for line in file.readlines():
            # Rechercher les correspondances dans chaque ligne
            #print(line)
            match = re.match(pattern, line)
            if match:
                # Ajouter les correspondances à la liste
                matches.append(match.groupdict())
            #print(matches)
        df = pd.DataFrame(matches)
        # Afficher le dataframe
        print(df)"""

Jun 18 00:03:42 srvpx01 sshd[3116664]: pam_unix(sshd:auth): authentication failure; logname= uid=0 euid=0 tty=ssh ruser= rhost=188.166.52.232  user=root

{'Timestamp': 'Jun 18 00:03:42', 'Node': 'srvpx01', 'Event': 'sshd', 'sshd_id': '3116664', 'common_messages': 'pam_unix(sshd:auth)', 'type_session': None, 'type_accepted': None, 'type_received': None, 'type_connection': None, 'type_fail': None, 'type_auth': 'failure', 'user': 'root', 'IP_user': None, 'port_user': None, 'logname': '', 'uid_target': None, 'uid_init': '0', 'euid': '0', 'tty': 'ssh', 'ruser': '', 'rhost': '188.166.52.232', 'others': None}
Jun 18 00:03:44 srvpx01 sshd[3116664]: Failed password for root from 188.166.52.232 port 46228 ssh2

{'Timestamp': 'Jun 18 00:03:44', 'Node': 'srvpx01', 'Event': 'sshd', 'sshd_id': '3116664', 'common_messages': 'Failed', 'type_session': None, 'type_accepted': None, 'type_received': None, 'type_connection': None, 'type_fail': None, 'type_auth': None, 'user': None, 'IP_user': None, 'port_u

KeyboardInterrupt: 

pam_unix session

In [ ]:
pattern = r'(\w+\s\d+\s\d{2}:\d{2}:\d{2})\s(\w+)\s(\w+)(\[(\d+)\]|):\s(\w+)\((\w+)\:(\w+)\):\s\w+\s(\w+)\s\w+\s\w+\s(\w+)\((\w+=\d+)\)\s\w+\s\((\w+=\d+)\)'

#\sfor\suser\s(\w+)\((\w+=\d+)\)\sby\s\((\w+=\d+)\)
# Exemple d'utilisation
log_line = 'Jun 23 15:38:15 srvpx02 sshd[622247]: pam_unix(sshd:session): session opened for user root(uid=0) by (uid=0)'

matchu = re.search(pattern, log_line)
if matchu:
    timestamp = matchu.group(1)
    hostname = matchu.group(2)
    process_name = matchu.group(3)
    process_id = matchu.group(5)
    session_info = matchu.group(6)
    action_1 = matchu.group(7)
    action_2 = matchu.group(8)
    action_3 = matchu.group(9)
    username = matchu.group(10)
    user_info = matchu.group(11)
    initiator_info = matchu.group(12)
    print("Timestamp:", timestamp)
    print("Hostname:", hostname)
    print("Process Name:", process_name)
    print("Process ID:", process_id)
    print("Session Info:", session_info)
    print("Action 1:", action_1)
    print("Action 2:", action_2)
    print("Action 3:", action_3)
    print("Username:", username)
    print("User info:", user_info)
    print("Initiator Info:", initiator_info)

Timestamp: Jun 23 15:38:15
Hostname: srvpx02
Process Name: sshd
Process ID: 622247
Session Info: pam_unix
Action 1: sshd
Action 2: session
Action 3: opened
Username: root
User info: uid=0
Initiator Info: uid=0


In [ ]:
common_messages = [
    r'Accepted',
    r'pam_unix\(.+\): session (?:opened|closed) for user .+',
    r'Failed',
    r'Invalid user .+',
    r'Authentication failure',
    r'refused connect from .+',
    r'error: maximum authentication attempts exceeded',
    r'Connection closed by .+',
    r'useradd',
    r'userdel',
    r'passwd',
]



# Concaténation des messages en une seule expression régulière
pattern = r'({})'.format('|'.join(common_messages))

# Exemple d'utilisation avec un fichier auth.log
with open('auth.log', 'r') as file:
    for line in file.readlines():
        match = re.search(pattern, line)
        if match:
            print("Matched message:", match.group(0))

FileNotFoundError: [Errno 2] No such file or directory: 'auth.log'